## Lets go pro build a rag agent without using langchain libary

In [50]:
import os
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import json
import re
from typing import Optional
import concurrent.futures
from functools import partial 

In [51]:
load_dotenv(override=True)

MODEL = "gemini-2.5-flash"
gemini_api_key = os.getenv("GEMINI_API_KEY")
base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

MODEL_OLLAMA = "llama3.2"
base_url_ollama = "http://localhost:11434/v1"

DB_NAME = "preprocesses_db"
collection_name = "docs"

embedding_model = "intfloat/multilingual-e5-large"
KNOWLEDGE_BASE_PATH = Path("rag_documents")

AVERAGE_CHUNK_SIZE = 500
openai = OpenAI()

In [52]:
class Result(BaseModel):
     page_content: str
     metadata: dict

In [53]:
class Chunk(BaseModel):
     headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
     summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
     original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

     def as_result(self, document):
          metadata = {"source": document["source"], "type": document["type"]}
          return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text, metadata=metadata) 
     
class Chunks(BaseModel):
     chunks: list[Chunk]

## Three steps 
### 1.) Fetch documents from the knowledge base 
### 2.) Call LLM to turn documents into chunks
### 3.) Store the chunks in chroma

In [54]:
def fetch_documents():
     """A Homemade function to fetch documents from the knowledge base"""
     
     documents = []
     
     for folder in KNOWLEDGE_BASE_PATH.iterdir():
          doc_type = folder.name
          for file in folder.rglob("*.md"):
               with open(file, "r", encoding="utf-8") as f:
                    documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})
                    
     print(f"Found {len(documents)} documents in the knowledge base")
     return documents

In [55]:
documents = fetch_documents()

Found 1041 documents in the knowledge base


In [56]:
documents[0]["text"]

'---\ndoc_type: customer\ndoc_id: customer_101\ncustomer_id: 101\ncustomer_name: (101)TIGER Lift & Handels GmbH\nmarket_type: Yurtdisi\ndelivery_time: 14\n---\n\nMÜŞTERİ BİLGİSİ: (101)TIGER Lift & Handels GmbH\n\n- Müşteri ID: 101\n- Müşteri Adı: (101)TIGER Lift & Handels GmbH\n- Libal Müşteri Kodu: 101\n- Pazar: Yurtdisi\n- Teslimat Süresi: 14\n- Sevkiyat Günü: 5'

## WOHOO Lets create chunks

In [57]:
def clean_json_string(json_str: str) -> str:
    """Clean JSON string to fix Unicode and formatting issues."""
    # Remove any markdown code blocks if present
    json_str = re.sub(r'```json\s*', '', json_str)
    json_str = re.sub(r'```\s*', '', json_str)
    json_str = json_str.strip()
    
    # Fix common Unicode issues - replace lone surrogates
    try:
        # Try to encode/decode to catch encoding issues
        json_str = json_str.encode('utf-8', errors='replace').decode('utf-8')
    except:
        pass
    
    # Try to extract JSON if wrapped in other text
    json_match = re.search(r'\{.*\}', json_str, re.DOTALL)
    if json_match:
        json_str = json_match.group(0)
    
    return json_str

def make_prompt(document):
     how_many = ((len(document["text"])) // AVERAGE_CHUNK_SIZE) + 1
     
     # Truncate very long documents to avoid token limits (llama3.2 8b has ~8k context)
     max_doc_length = 6000  # Leave room for prompt and response
     doc_text = document["text"]
     if len(doc_text) > max_doc_length:
         doc_text = doc_text[:max_doc_length] + "\n\n[Document truncated due to length]"
     
     return f"""You are a document processing assistant. Split the following document into overlapping chunks for a knowledge base.

Document type: {document["type"]}
Source: {document["source"]}
Estimated chunks needed: {how_many}

INSTRUCTIONS:
1. Split the document into {how_many} chunks (more or less as needed)
2. Ensure 25% overlap between chunks (about 50 words)
3. For each chunk, provide:
   - headline: A brief heading (3-5 words) that would match search queries
   - summary: 2-3 sentences summarizing the chunk content
   - original_text: The exact original text from the document

CRITICAL: Return ONLY valid JSON in this exact format (no markdown, no code blocks, no extra text):
{{
  "chunks": [
    {{
      "headline": "Brief heading",
      "summary": "Summary text",
      "original_text": "Original text content"
    }}
  ]
}}

- Ensure all text is properly escaped for JSON
- Make sure the entire document is covered by chunks
- Do NOT include any explanations or markdown formatting

Document:
{doc_text}

Return the JSON response now:"""

In [58]:
print(make_prompt(documents[0]))

You are a document processing assistant. Split the following document into overlapping chunks for a knowledge base.

Document type: customers
Source: rag_documents/customers/(101)TIGER_Lift_&_Handels_GmbH.md
Estimated chunks needed: 1

INSTRUCTIONS:
1. Split the document into 1 chunks (more or less as needed)
2. Ensure 25% overlap between chunks (about 50 words)
3. For each chunk, provide:
   - headline: A brief heading (3-5 words) that would match search queries
   - summary: 2-3 sentences summarizing the chunk content
   - original_text: The exact original text from the document

CRITICAL: Return ONLY valid JSON in this exact format (no markdown, no code blocks, no extra text):
{
  "chunks": [
    {
      "headline": "Brief heading",
      "summary": "Summary text",
      "original_text": "Original text content"
    }
  ]
}

- Ensure all text is properly escaped for JSON
- Make sure the entire document is covered by chunks
- Do NOT include any explanations or markdown formatting

Doc

In [59]:
def make_messages(document):
     return [{"role": "user", "content": make_prompt(document)}]

In [60]:
make_messages(documents[0])

[{'role': 'user',
  'content': 'You are a document processing assistant. Split the following document into overlapping chunks for a knowledge base.\n\nDocument type: customers\nSource: rag_documents/customers/(101)TIGER_Lift_&_Handels_GmbH.md\nEstimated chunks needed: 1\n\nINSTRUCTIONS:\n1. Split the document into 1 chunks (more or less as needed)\n2. Ensure 25% overlap between chunks (about 50 words)\n3. For each chunk, provide:\n   - headline: A brief heading (3-5 words) that would match search queries\n   - summary: 2-3 sentences summarizing the chunk content\n   - original_text: The exact original text from the document\n\nCRITICAL: Return ONLY valid JSON in this exact format (no markdown, no code blocks, no extra text):\n{\n  "chunks": [\n    {\n      "headline": "Brief heading",\n      "summary": "Summary text",\n      "original_text": "Original text content"\n    }\n  ]\n}\n\n- Ensure all text is properly escaped for JSON\n- Make sure the entire document is covered by chunks\n- 

In [45]:
gemini = OpenAI(api_key=gemini_api_key, base_url=base_url)
ollama = OpenAI(base_url=base_url_ollama, api_key="ollama")

In [61]:
def process_document(document, max_retries=2):
    """Process a document with retry logic and error handling."""
    messages = make_messages(document)
    
    for attempt in range(max_retries):
        try:
            # Use regular completion with JSON mode instead of parse for better control
            response = ollama.chat.completions.create(
                model=MODEL_OLLAMA, 
                messages=messages,
                temperature=0.1,  # Lower temperature for more consistent JSON
                response_format={"type": "json_object"}  # Force JSON mode
            )
            
            reply = response.choices[0].message.content
            
            # Clean the JSON response
            cleaned_json = clean_json_string(reply)
            
            # Parse with error handling
            try:
                parsed = Chunks.model_validate_json(cleaned_json)
                doc_as_chunks = parsed.chunks
                return [chunk.as_result(document) for chunk in doc_as_chunks]
            except Exception as e:
                # If validation fails, try to fix common issues
                if attempt < max_retries - 1:
                    # Try to extract chunks manually from JSON
                    try:
                        json_obj = json.loads(cleaned_json)
                        if "chunks" in json_obj:
                            chunks_data = json_obj["chunks"]
                            # Reconstruct Chunk objects
                            doc_chunks = []
                            for chunk_data in chunks_data:
                                # Ensure all required fields exist
                                chunk = Chunk(
                                    headline=chunk_data.get("headline", "Document Content"),
                                    summary=chunk_data.get("summary", ""),
                                    original_text=chunk_data.get("original_text", "")
                                )
                                doc_chunks.append(chunk)
                            return [chunk.as_result(document) for chunk in doc_chunks]
                    except Exception as json_err:
                        if attempt == max_retries - 1:
                            print(f"⚠️  JSON parse failed for {document['source'][:50]}...: {str(json_err)[:80]}")
                
                if attempt == max_retries - 1:
                    # Last attempt failed - return minimal chunk
                    minimal_chunk = Chunk(
                        headline="Document Content",
                        summary=document["text"][:200] + "...",
                        original_text=document["text"]
                    )
                    return [minimal_chunk.as_result(document)]
                
        except Exception as e:
            if attempt == max_retries - 1:
                # Return minimal chunk on final failure
                minimal_chunk = Chunk(
                    headline="Document Content",
                    summary=document["text"][:200] + "...",
                    original_text=document["text"]
                )
                return [minimal_chunk.as_result(document)]
    
    return []

In [62]:
process_document(documents[0])

[Result(page_content='Customer Information\n\nContains customer details such as ID, name, and market type.\n\n&lt;span style="font-size:12pt; color:#000000; font-family:Arial; font-weight:bold;&gt;MüÅŸteri Bılgisi:&lt;/span&gt; \\n\\n- Çustomer ID: 101\\n- Çustomer Adı: \\u00f6zgúr Lift &amp; Handels GmbH\\n- Libal Ömleti Kodu: 101\\n- Pazar: Yurtdisi\\n- Teslimat Öresi: 14\\n- Sevkiyat Günu: 5', metadata={'source': 'rag_documents/customers/(101)TIGER_Lift_&_Handels_GmbH.md', 'type': 'customers'})]

In [63]:
def create_chunks(documents, max_workers=3, batch_size=20):
    """
    Create chunks from documents with parallel processing and batching.
    
    Args:
        documents: List of documents to process
        max_workers: Number of parallel workers (3 is safe for llama3.2 8b)
        batch_size: Process in batches to track progress better
    """
    chunks = []
    total_docs = len(documents)
    
    # Process in batches to avoid overwhelming the system and track progress
    for batch_start in tqdm(range(0, total_docs, batch_size), desc="Processing batches"):
        batch = documents[batch_start:batch_start + batch_size]
        
        # Process batch with parallel workers
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_doc = {executor.submit(process_document, doc): doc for doc in batch}
            
            for future in concurrent.futures.as_completed(future_to_doc):
                doc = future_to_doc[future]
                try:
                    doc_chunks = future.result()
                    chunks.extend(doc_chunks)
                except Exception as e:
                    # Add minimal chunk on error
                    minimal_chunk = Chunk(
                        headline="Document Content",
                        summary=doc["text"][:200] + "...",
                        original_text=doc["text"]
                    )
                    chunks.append(minimal_chunk.as_result(doc))
    
    print(f"\n✅ Processed {total_docs} documents into {len(chunks)} chunks")
    return chunks

In [ ]:
chunks = create_chunks(documents)

Processing batches:   0%|          | 0/53 [00:00<?, ?it/s]

### Day 5 of the AI engineer core track i left at the day 131 i need a more powerfull pc to continue maybe a rtx4080 would do it